In [4]:
import os
from elasticsearch import Elasticsearch
import json
import pandas as pd

from extraction_prompts import (
    EXTRACT_RELATION_TRIPLETS_PROMPT,
    REFORMULATE_RELATION_TRIPLET,
    VALIDATE_TRIPLET_USEFULNESS,

)
from utils import ollama_request

ES_HOST = os.getenv("ES_HOST", "http://localhost:9200")
INDEX_NAME = os.getenv("INDEX_NAME")
FRAGMENT_INDEX_NAME = f"{os.getenv('INDEX_NAME')}_fragments"
es_client = Elasticsearch('http://localhost:9200')


In [2]:
def create_relations(es_client, fragment_index_name, step=2, reformulate_predicate=True, n_speeches=100):
    final_triplets = pd.DataFrame(columns=["speech_id", "start", "end", "date", "subject", "predicate", "object"])
    speech_counter = 1
    for speech_id in range(1, n_speeches+2):
        retrieve_attempts = 0
        while True:
            retrieve_attempts += 1
            if retrieve_attempts > 5:
                break
            fragments_for_text = es_client.search(
                index=fragment_index_name,
                query={
                    "match": {
                        "speech_id": speech_counter
                    }
                },
                size=10000
            )
            hits = fragments_for_text['hits']['hits']
            speech_counter += 1
            if len(hits) == 0:
                continue
            else:
                break
        if len(hits) == 0:
            break
        if speech_counter > n_speeches+2:
            break
        print(f"Processing speech_id: {speech_id}, number of fragments: {len(hits)}")
        for fragment_id in range(0, len(hits), step):
            fragment_group = hits[fragment_id:fragment_id + step]
            fragment_text = " ".join(
                hit["_source"]["text"]
                for hit in fragment_group
                if hit["_source"].get("text")
            )
            jsoned_triplets = None
            for attempt in range(2):
                triplets = ollama_request(
                    prompt=EXTRACT_RELATION_TRIPLETS_PROMPT.format(fragment=fragment_text, speech_author=fragment_group[0]["_source"].get("author", "Unknown")),
                    is_stream=False
                ).replace("```json", "").replace("```", "").replace("*", "").strip()

                try:
                    jsoned_triplets = json.loads(triplets)["triplets"]
                    break
                except json.JSONDecodeError as e:
                    if attempt != 0:
                        continue
                    # else:
                    #     print(f"JSON decoding error for speech_id {speech_id}, fragment_id {fragment_id}, Retrying once with the same prompt...")

            if jsoned_triplets is None:
                continue

            for triplet in jsoned_triplets:
                if triplet is None or not isinstance(triplet, dict):
                    continue
                subject = triplet.get("subject", None)
                predicate = triplet.get("predicate", None)
                object_ = triplet.get("object", None)

                if object_:
                    object_ = str(object_).strip()

                if any(not x for x in [subject, predicate, object_]):
                    continue

                reformulated_triplet = {}
                if reformulate_predicate:
                    reformulated_predicate = ollama_request(
                            prompt=REFORMULATE_RELATION_TRIPLET.format(subject=subject, predicate=predicate, object=object_),
                            is_stream=False
                    ).replace("```json", "").replace("```", "").replace("*", "").strip()
                    # print(f"Changed {predicate} to: {reformulated_predicate}")
                    reformulated_triplet["predicate"] = reformulated_predicate

                reformulated_triplet = {
                    "subject": reformulated_triplet.get("subject", subject).split("(")[0].strip(),
                    "predicate": reformulated_triplet.get("predicate", predicate),
                    "object": reformulated_triplet.get("object", object_).split("(")[0].strip()
                }
                
                is_valid = ollama_request(
                    prompt=VALIDATE_TRIPLET_USEFULNESS.format(
                        subject=reformulated_triplet.get("subject", subject),
                        predicate=reformulated_triplet.get("predicate", predicate),
                        object=reformulated_triplet.get("object", object_)
                    ),
                    is_stream=False
                )
                if is_valid.strip().lower().replace('"', "").replace("'", "") != "useful":
                    # print(f"Triplet: {reformulated_triplet.get('subject', subject)}, {reformulated_triplet.get('predicate', predicate)}, {reformulated_triplet.get('object', object_)} deemed not useful")
                    continue
                final_triplets = pd.concat([final_triplets, pd.DataFrame([{
                    "speech_id": speech_counter,
                    "start": fragment_group[0]["_source"]["chunk_start"],
                    "end": fragment_group[-1]["_source"]["chunk_end"],
                    "date": fragment_group[0]["_source"]["date"],
                    "subject": reformulated_triplet.get("subject", subject).lower().strip(),
                    "predicate": reformulated_triplet.get("predicate", predicate).lower().strip(),
                    "object": reformulated_triplet.get("object", object_).lower().strip()
                }])], ignore_index=True)
    return final_triplets

In [3]:
triplets = create_relations(es_client, FRAGMENT_INDEX_NAME, step=2, reformulate_predicate=True, n_speeches=30)
triplets.to_csv("data/triplets.csv", index=False)

Processing speech_id: 1, number of fragments: 16
Processing speech_id: 2, number of fragments: 19
Processing speech_id: 3, number of fragments: 65
Processing speech_id: 4, number of fragments: 9
Processing speech_id: 5, number of fragments: 80
Processing speech_id: 6, number of fragments: 18
Processing speech_id: 7, number of fragments: 26
Processing speech_id: 8, number of fragments: 60
Processing speech_id: 9, number of fragments: 92
Processing speech_id: 10, number of fragments: 54
Processing speech_id: 11, number of fragments: 9
Processing speech_id: 12, number of fragments: 34
Processing speech_id: 13, number of fragments: 5
Processing speech_id: 14, number of fragments: 30
Processing speech_id: 15, number of fragments: 25
Processing speech_id: 16, number of fragments: 8
Processing speech_id: 17, number of fragments: 143
Processing speech_id: 18, number of fragments: 60
Processing speech_id: 19, number of fragments: 80
Processing speech_id: 20, number of fragments: 11
Processing s